# 04 · Fields and samplers

A **sampler** maps events $x = (a, x, y, z)$ (shape `(..., 4)`, comoving Mpc) to the value of a gas field there. For
every absorbing species the optical depth needs five fields (review notes, Sec. GasProperties):

| field | unit | kind |
|---|---|---|
| `temperature` | K | scalar |
| `velocity` | km/s (peculiar) | vector `(..., 3)` |
| `number_density` | comoving cm$^{-3}$ | scalar |
| `turbulence` | m/s ($b_{\rm turb}$) | scalar |
| `collisional` | GHz ($A_{\rm coll}$) | scalar |

`ScalarField` and `VectorField` wrap any callable, and `Fields({species: {name: field}})` holds them all. The core ships
these samplers:

- `LogRamp`, `VelocityRamp` and `Constant`, simple analytic fields;
- `GRFDensity`, a density $n = \bar n\,(1 + \delta)$ from one `PowerLawGRF` realisation $\delta$;
- `SampledField`, values already known at the points of a skewer bundle. This is how simulation outputs enter (05).

All of them are JAX pytrees. This notebook covers the analytic samplers first, then Gaussian random fields. Runtime:
~1 min.

**Requires:** nothing: samplers are functions of events $(a, x, y, z)$.  
**Independent of:** lines, cosmology.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('skewers', 'samplers', 'fields',)); plt.show()

In [ ]:
from skewerjax.core import (Fields, ScalarField, VectorField, SampledField, LogRamp, VelocityRamp, Constant, PowerLawGRF,
                            GRFDensity, plane_points)
L = 1.0
species = {"temperature": ScalarField(LogRamp(1e4, 1e6, L, axis=3)),          # log ramp along z
           "velocity": VectorField(VelocityRamp(-50.0, 80.0, L)),             # v_x linear in x
           "number_density": ScalarField(LogRamp(1e-9, 1e-3, L, axis=2)),     # log ramp along y
           "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
fields = Fields({"HI": species, "SiIII": species})
pts = plane_points(L, "x", 0.5, 64, 64)                                        # (64, 64, 3) on the plane x = 0.5
events = jnp.concatenate([jnp.full(pts.shape[:-1] + (1,), 0.9), pts], -1)       # a = 0.9
directions = jnp.broadcast_to(jnp.array([1.0, 0.0, 0.0]), pts.shape)            # photons along +x
sampled = fields.sample(events, directions)
print({sp: {k: v.shape for k, v in d.items()} for sp, d in sampled.items()})

`sample` adds `LoS_velocity`, the projection of the velocity on the photon direction,
$v_\parallel = \mathbf{v}\cdot\hat d$. Called on several species at once, `fields.temperature(x)` stacks them, with
shape `(N_species, ...)`.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 6), layout="constrained")
ext = (0, L, 0, L)
for ax, (name, cmap) in zip(axs[:2], (("temperature", "incandescent"), ("number_density", "iridescent"))):
    from skewerjax.utils.plotting import tol_cmap
    im = ax.imshow(np.log10(np.asarray(sampled["HI"][name])).T, origin="lower", extent=ext, cmap=tol_cmap(cmap))
    fig.colorbar(im, ax=ax, label=f"log10 {name}")
    ax.set(xlabel="y [cMpc]", ylabel="z [cMpc]", title=f"LogRamp: {name}")
xs = jnp.linspace(0, L, 50)
ev_x = jnp.stack([jnp.full(50, 0.9), xs, jnp.full(50, 0.5), jnp.full(50, 0.5)], -1)
axs[2].plot(xs, fields.LoS_velocity(ev_x, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (50, 3)))[0], color=EC["velocity"])
axs[2].set(xlabel="x [cMpc]", ylabel=r"$v_\parallel$ [km/s]", title="VelocityRamp along x, projected on +x")
add_panel_letters(axs); plt.show()

## 1 · Gaussian random fields: `PowerLawGRF`

`PowerLawGRF` draws a periodic Gaussian random field in 1, 2 or 3 dimensions, with spectrum

$$
P(k) = A_s \left(\frac{k}{k_0}\right)^{n_s - 1} .
$$

The grid is given either by $(L, N)$ or by two of $(k_{\min}, k_{\max}, N_k)$. The normalisation is that of the P1D
estimator of the review notes, $\langle \lvert\delta_k\rvert^2\rangle / V = P(k)$, so `variance()` returns the
expected $\sigma^2$. `sample(key)` is jitted, and `jax.vmap` over keys draws a batch.

In [ ]:
key = jax.random.PRNGKey(1)
g1 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-0.5, L=100.0, N=1024)
g2 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-1.5, L=(50.0, 100.0), N=(128, 256))     # anisotropic box: y = 2 x
g3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, k_min=(2 * np.pi / 50,) * 3, N_k=(256,) * 3)   # mode (b): N_k modes per axis
for g in (g1, g2, g3):
    print(g, f"sigma = {float(jnp.sqrt(g.variance())):.3f}")
d1, d2, d3 = g1.sample(key), g2.sample(key), g3.sample(key)
iz = d3.shape[2] // 2                                                              # the slice z = L/2
fig, axs = plt.subplots(1, 3, figsize=(24, 7.5), layout="constrained", gridspec_kw={"width_ratios": [1.3, 0.75, 1]})
axs[0].plot(g1.x_grid()[0], d1, color=EC["density"], lw=1); axs[0].set(xlabel="x [Mpc]", ylabel=r"$\delta$", title="1D, $n_s = -0.5$")
Lx, Ly = g2.L
im = axs[1].imshow(np.asarray(d2).T, origin="lower", extent=(0, Lx, 0, Ly), cmap=CMAPS["diverging"], aspect="equal")
axs[1].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title="2D, box 50 × 100 Mpc")
L3 = g3.L[0]
axs[2].imshow(np.asarray(d3[:, :, iz]).T, origin="lower", extent=(0, L3, 0, L3), cmap=CMAPS["diverging"], aspect="equal")
axs[2].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title=f"3D, {d3.shape[0]}³ cells, slice z = L/2 (index {iz})")
add_panel_letters(axs); plt.show()

The measured power spectrum of many draws recovers the input in every dimension. The estimator is the one that
defines the normalisation:

$$
\hat P(k) = \frac{\lvert\delta_k\rvert^2}{V},
$$

where $\delta_k$ is the discrete Fourier transform times the cell volume. It is averaged over the modes of each bin and
over the draws; the error bars are the scatter between draws.

- **Isotropic $P(\lvert k\rvert)$** (top two rows): every mode in a bin of $\lvert k\rvert$, in 1D, 2D and 3D,
  against the input, with residuals.
- **The $(k_\parallel, k_\perp)$ plane** (bottom row, 2D and 3D): the measured power, as colour, in bins of the
  wavenumber along the last axis, $k_\parallel$, and across it, $k_\perp$. The dashed curves are iso-power contours
  of the input on the same colour levels.
  - Both axes are logarithmic (one mode per bin at low $k$, logarithmic bins above), so the modes with
    $k_\parallel = 0$ or $k_\perp = 0$ are not shown.
  - For an isotropic spectrum the measured colours follow the curves $k_\parallel^2 + k_\perp^2 = \text{const}$
    traced by the input contours. This is what justifies the isotropic average above.
  - The anisotropic 2D box ($L_y = 2 L_x$) has different fundamental modes along the two directions, $2\pi/100$ and
    $2\pi/50\ \mathrm{Mpc^{-1}}$, and it still follows one $P(k)$. Its bins are as wide as the coarser of the two.

Full checks: `scripts/validation/check_summary_statistics.py`.

In [ ]:
from skewerjax.utils.plotting import plot_pk_recovery_nd
draws = lambda g, n, seed: np.asarray(jax.vmap(g.sample)(jax.random.split(jax.random.PRNGKey(seed), n)))
cases = {"1D, 200 draws": (g1, draws(g1, 200, 2)), "2D (50 × 100 Mpc), 40 draws": (g2, draws(g2, 40, 3)),
         "3D (512³), 4 draws": (g3, draws(g3, 4, 4))}
fig, axs, rec = plot_pk_recovery_nd(cases)
add_panel_letters(axs.ravel()[[0, 1, 2, 3, 4, 5, 7, 8]])
fig.suptitle("PowerLawGRF: the measured P(k) is the input, isotropically (top) and in the (k∥, k⊥) plane (bottom)", fontsize=22)
plt.show()
for name, r in rec.items():
    extra = f", (k_par, k_perp) map chi2/dof = {r['map_chi2_dof']:.2f}" if "map_chi2_dof" in r else ""
    print(f"{name:28s}: isotropic P(|k|), chi2/dof = {r['chi2_dof']:.2f}, max |P / P_in - 1| = {r['max_rel']:.3f}{extra}")

**Densities from a GRF.** `GRFDensity(key, grf, n_bar)` turns one 3D realisation into a density,

$$
n(\mathbf{x}) = \bar n\,\big[1 + \delta(\mathbf{x})\big],
$$

interpolated trilinearly in the periodic box, with no other transform. The overdensity $n/\langle n\rangle - 1 = \delta$
therefore has exactly the input spectrum on the grid.

- $\bar n$ is the mean density, because $\delta$ has no $k = 0$ mode.
- The amplitude $A_s$ must keep $1 + \delta > 0$. `min_one_plus_delta` reports the minimum, and a warning is issued
  otherwise; nothing is clipped, since clipping would change the statistics.
- `delta_grid=` reuses one realisation for several species.

Notebooks 00, 06 and 07 build their toy densities this way.

## 2 · Fields are pytrees

A **pytree** is JAX's name for a nested container whose leaves are arrays: tuples, lists, dicts, and any class
registered with `jax.tree_util` ([JAX documentation: Pytrees](https://docs.jax.dev/en/latest/pytrees.html)). JAX's
transformations accept pytrees as inputs and outputs. `jax.jit` compiles a function of a whole `Fields` object, and
`jax.vmap` and `jax.grad` map over, or differentiate with respect to, the arrays inside it. All of this needs no change
to the function.

Every sampler, `ScalarField`, `VectorField` and `Fields` is registered as a pytree:

- **leaves:** the arrays and numbers they hold (ramp limits, a GRF realisation, sampled values);
- **static structure:** everything else (which species, which field, the axis of a ramp). Changing the structure
  recompiles; changing a leaf does not.

`jax.tree_util.tree_map` builds a new `Fields` with transformed leaves, for example every parameter scaled. Plain
Python callables wrapped in a `ScalarField` are kept as static data, so closures work too.

In [ ]:
f_jit = jax.jit(lambda f, x, d: f.sample(x, d))
out = f_jit(fields, events, directions)
diff = max(float(jnp.max(jnp.abs(out[sp][k] - sampled[sp][k]) / jnp.maximum(jnp.abs(sampled[sp][k]), 1e-300))) for sp in sampled for k in sampled[sp])
leaves, treedef = jax.tree_util.tree_flatten(fields)
print(f"the Fields pytree: {len(leaves)} leaves, e.g. {[float(x) for x in leaves[:3]]} (the T ramp's v_min, v_max, L_box)")
print(f"jit(sample) vs sample: max relative difference {diff:.1e} (rounding)")
hot = jax.tree_util.tree_map(lambda x: x * 2.0, fields)          # a new Fields with every leaf doubled, same structure
print("tree_map(2 x leaves): T at the first event", float(fields.sample(events, directions)["HI"]["temperature"][0, 0]),
      "->", float(hot.sample(events, directions)["HI"]["temperature"][0, 0]))